# FreshCart — Snowflake ELT Pipeline

**Student:** Prudhvi Raj Kumar  
**Roll number:** 23BCS209  
**Logical run date:** 2026-09-04

Reconstructed notebook documenting the preserved FreshCart ELT implementation and its execution evidence. The original course notebook was not present in the supplied source ZIP or repository.

## 1. Provenance and execution boundaries

This notebook is reconstructed from the preserved Python and SQL source files and saved evidence. It is not the missing original course notebook.

The local dry run recorded below is a genuine extraction-only execution. It wrote compressed landing files and did not touch Snowflake. The Snowflake transformation results in later sections are historical results from earlier recorded runs, not output of that dry run.

## 2. ELT architecture

```text
Web / Mobile / CRM / FX APIs
            |
         Extract
            |
     Compressed landing
            |
         BRONZE
            |
    SILVER validation
       /         \
    Clean      Quarantine
       |
 Deduplication + SCD2
 FX + Pincode references
       |
      GOLD
       |
 Quality checks
```

## 3. Inspect the preserved source files

The executable implementation remains in `elt/`. Run the following cell from the repository root or from the `elt/` directory.

In [ ]:
from pathlib import Path

elt = Path.cwd()
if not (elt / "run_pipeline.py").exists():
    elt = elt / "elt"

print("Python source files:")
for p in sorted(elt.glob("*.py")):
    print("-", p.name)

print("\nSQL transformation files:")
for p in sorted((elt / "sql").glob("*.sql")):
    print("-", p.name)


## 4. Local extraction dry run

Command executed:

```powershell
py -3.12 .\elt\run_pipeline.py --dry-run --mock --run-date 2026-09-04
```

The following is the captured output from that successful run. It is included as recorded output and is not claimed as a fresh execution of this notebook.

In [1]:
print('Captured terminal output from the successful local dry run.')

07:28:31 pipeline  run_id=20260904_015831_907808 run_date=2026-09-04 mock=True
07:28:31 extract   web: 6 records -> landing\web\2026-09-04\web__20260904_015831_907808.json.gz
07:28:31 extract   mobile: 2 records -> landing\mobile\2026-09-04\mobile__20260904_015831_907808.json.gz
07:28:31 extract   crm: 5 records -> landing\crm\2026-09-04\crm__20260904_015831_907808.json.gz
07:28:31 extract   fx: 1 records -> landing\fx\2026-09-04\fx__20260904_015831_907808.json.gz
07:28:31 pipeline  dry run: landing files written, Snowflake not touched.

### Extraction summary

| Source | Records |
|---|---:|
| Web orders | 6 |
| Mobile orders | 2 |
| CRM | 5 |
| FX | 1 |
| **Total** | **14** |

The run ID was `20260904_015831_907808`. The final log line explicitly confirms that Snowflake was not touched.

## 5. SQL transformation layers

| SQL file | Purpose |
|---|---|
| `00_setup.sql` | Create Snowflake objects |
| `10_silver_orders_clean.sql` | Normalize, validate and quarantine order records |
| `20_silver_orders_dedup.sql` | Deduplicate orders |
| `30_dim_customers_scd2.sql` | Build customer history with SCD Type 2 |
| `31_ref_fx_daily.sql` | Prepare daily FX reference |
| `32_ref_pincode_geo.sql` | Prepare pincode geography reference |
| `40_gold_fct_orders.sql` | Build Gold order fact table |

The SQL source files remain the implementation source of truth.

## 6. Part B1 — Baseline results

Previously recorded Snowflake baseline:

- Gold orders: **4**
- Gold revenue: **₹7,785**
- Loyalty points: **126**
- Bronze order rows per run: **8**
- Silver accepted rows: **6**
- Silver quarantined rows: **2**

Two append runs produced 16 Bronze order rows. The baseline CRM fixture and later B4 CRM event fixture represent different database states and should not be interpreted as simultaneous results.

## 7. Part B2 — Order tracing

**ORD-1003:** The later mobile record superseded the earlier web record. The selected amount was ₹2,100. Meera's Platinum multiplier of 3 produced 63 points:

`FLOOR(2100 / 100) × 3 = 63`

**ORD-1005:** Priya's Gold tier was effective 2026-09-03. For ₹899, the Gold multiplier of 2 produced 16 points:

`FLOOR(899 / 100) × 2 = 16`

These are recorded Snowflake transformation results, not outputs from the extraction-only dry run.

## 8. Part B3 — Validation and quarantine

The Silver validation logic checks order ID, email, pincode, dates, positive amount and supported currency.

ORD-1006 used unsupported EUR currency. Its recorded validation reasons after the currency check were:

- `bad_email`
- `bad_pincode`
- `unsupported_currency`

It was excluded from Silver clean data and Gold.

Evidence files are in the repository's `evidence/` directory:

- `B3_ORD1006_validation.png`
- `B3_quarantine_duplicates.png`
- `B3_quarantine_json_case_mismatch.png`
- `B3_gold_exclusion.png`

The investigation found two quarantine records for ORD-1006 with different run IDs. The JSON key was stored as uppercase `UPDATED_AT`, while a lowercase JSON path was used in the duplicate comparison. This documents the observed cause; it does not claim the SQL issue was repaired.

## 9. Part B4 — Customer SCD Type 2

The recorded CUST-02 history shows the earlier Silver tier ending on 2026-08-14 and Gold becoming effective on 2026-08-15.

In the extended CRM scenario, ORD-1002 had USD 42.50, converted to ₹3,536, with Gold multiplier 2 and 70 loyalty points. The extended scenario total was 161 points; the baseline total was 126.

Relevant saved evidence:

- `B4_CUST02_SCD2_history.png`
- `B4_ORD1002_gold_current.png`
- `B4_gold_merge_history.png`
- `B4_gold_row_counts.png`

The query history shows successful MERGE executions. It does not establish an exact `ROWS_UPDATED` count, so no such count is claimed here.

## 10. Part B5 — Quality check failure and recovery

A deliberate test inserted a temporary Gold row with a source amount but no INR conversion. The strict quality check failed as expected. The test row was removed, and the recovery check passed.

See `evidence/B5_gold_quality_check.txt`. This was a controlled quality-gate test, not a production order.

## 11. Reproducibility and limitations

1. Local extraction is reproducible with `--dry-run --mock`.
2. Full Snowflake execution requires configured credentials, permissions, warehouse and database objects.
3. The baseline CRM fixture and B4 CRM event fixture produce different customer-history states.
4. Saved screenshots and transcripts are historical execution evidence.
5. The original course notebook was unavailable; this notebook reconstructs documentation around the preserved source and evidence.
6. Never commit `.env` files, passwords, private keys or access tokens.

## 12. Conclusion

The FreshCart pipeline separates extraction, raw ingestion, Silver validation, deduplication, SCD2 customer history, reference data, Gold modelling and quality checks. The local dry run verifies extraction and landing-file creation; saved Snowflake evidence documents the earlier transformation and quality-check work.